In [1]:
import kagglehub
import os
import pandas as pd

# Датасет уже скачан, просто получаем путь
path = kagglehub.dataset_download("birdy654/deep-voice-deepfake-voice-recognition")
print("Путь к датасету:", path)

# Смотрим ЧТО ИМЕННО там лежит
print("\n=== Структура датасета ===")
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:
        print(f'{subindent}{file}')
    if len(files) > 5:
        print(f'{subindent}... и ещё {len(files)-5} файлов')

/Users/a/Desktop/Deepfake/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/a/Desktop/Deepfake/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 10%|█         | 385M/3.69G [22:05<3:14:27, 305kB/s] 


ChunkedEncodingError: ("Connection broken: ConnectionResetError(54, 'Connection reset by peer')", ConnectionResetError(54, 'Connection reset by peer'))

In [ ]:
import glob

# Ищем все CSV файлы
csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
print("CSV файлы:", csv_files)

# Ищем все аудиофайлы
audio_extensions = ['*.wav', '*.mp3', '*.flac', '*.ogg']
audio_files = []
for ext in audio_extensions:
    audio_files += glob.glob(os.path.join(path, '**', ext), recursive=True)

print(f"\nНайдено аудиофайлов: {len(audio_files)}")
print("Примеры путей:")
for f in audio_files[:5]:
    print(f" ", f)

In [ ]:
# ============================================================
# Строим DataFrame вручную из структуры папок
# Обычно структура: REAL/ и FAKE/ папки
# ============================================================

records = []
for audio_path in audio_files:
    # Определяем метку по имени папки
    parts = audio_path.lower().replace('\\', '/').split('/')

    if 'real' in parts or 'genuine' in parts or 'original' in parts:
        label = 'real'
    elif 'fake' in parts or 'spoof' in parts or 'synthetic' in parts or 'deepfake' in parts:
        label = 'fake'
    else:
        label = 'unknown'

    records.append({
        'filepath': audio_path,
        'filename': os.path.basename(audio_path),
        'label': label,
        'folder': os.path.dirname(audio_path).split('/')[-1]
    })

df = pd.DataFrame(records)

print("=== DataFrame создан ===")
print(f"Размер: {df.shape}")
print(f"\nРаспределение классов:\n{df['label'].value_counts()}")
print(f"\nУникальные папки:\n{df['folder'].value_counts()}")
print(f"\nПервые 5 строк:\n{df.head()}")

In [ ]:
# ============================================================
# STEP 1: Посмотрим на аудиофайлы подробнее
# ============================================================
import librosa
import numpy as np

# Добавляем метаданные аудио в DataFrame
durations = []
sample_rates = []

for filepath in df['filepath']:
    try:
        y, sr = librosa.load(filepath, sr=None)
        durations.append(len(y) / sr)
        sample_rates.append(sr)
    except Exception as e:
        print(f"Ошибка: {filepath} — {e}")
        durations.append(None)
        sample_rates.append(None)

df['duration'] = durations
df['sample_rate'] = sample_rates

print(df[['filename', 'label', 'duration', 'sample_rate']].to_string())

In [ ]:
# ============================================================
# STEP 2: Визуализация — сравниваем REAL vs FAKE
# ============================================================
import matplotlib.pyplot as plt
import librosa.display

# Берём по 1 примеру каждого класса
real_file = df[df['label'] == 'real']['filepath'].iloc[0]
fake_file = df[df['label'] == 'fake']['filepath'].iloc[0]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for i, (filepath, label) in enumerate([(real_file, 'REAL'), (fake_file, 'FAKE')]):
    y, sr = librosa.load(filepath, sr=None)

    # Waveform
    axes[i, 0].plot(np.linspace(0, len(y)/sr, len(y)), y,
                    color='steelblue' if label == 'REAL' else 'tomato', alpha=0.7)
    axes[i, 0].set_title(f'{label} — Waveform')
    axes[i, 0].set_xlabel('Время (сек)')

    # Mel-спектрограмма
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[i, 1])
    axes[i, 1].set_title(f'{label} — Mel-Spectrogram')
    plt.colorbar(img, ax=axes[i, 1], format='%+2.0f dB')

    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    img2 = librosa.display.specshow(mfcc, sr=sr, x_axis='time', ax=axes[i, 2])
    axes[i, 2].set_title(f'{label} — MFCC')
    plt.colorbar(img2, ax=axes[i, 2])

plt.tight_layout()
plt.savefig('eda_real_vs_fake.png', dpi=150)
plt.show()
print("Визуализация сохранена!")

In [ ]:
# ============================================================
# STEP 3: Извлечение признаков — Feature Engineering
# ============================================================

def extract_features(filepath):
    """Извлекаем все акустические признаки из аудиофайла"""
    try:
        y, sr = librosa.load(filepath, sr=16000, duration=10)  # фиксируем 10 сек
        features = {}

        # --- MFCC (13 коэффициентов x mean/std) ---
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        for i in range(13):
            features[f'mfcc_{i}_mean'] = np.mean(mfcc[i])
            features[f'mfcc_{i}_std']  = np.std(mfcc[i])

        # --- Mel-spectrogram statistics ---
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        features['mel_mean'] = np.mean(mel_db)
        features['mel_std']  = np.std(mel_db)
        features['mel_max']  = np.max(mel_db)
        features['mel_min']  = np.min(mel_db)

        # --- Spectral features ---
        spec_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        features['spectral_centroid_mean'] = np.mean(spec_centroid)
        features['spectral_centroid_std']  = np.std(spec_centroid)

        spec_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        features['spectral_rolloff_mean'] = np.mean(spec_rolloff)

        spec_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        features['spectral_bandwidth_mean'] = np.mean(spec_bandwidth)

        # --- Zero Crossing Rate ---
        zcr = librosa.feature.zero_crossing_rate(y)
        features['zcr_mean'] = np.mean(zcr)
        features['zcr_std']  = np.std(zcr)

        # --- RMS Energy ---
        rms = librosa.feature.rms(y=y)
        features['rms_mean'] = np.mean(rms)
        features['rms_std']  = np.std(rms)

        # --- Chroma ---
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        features['chroma_mean'] = np.mean(chroma)
        features['chroma_std']  = np.std(chroma)

        return features

    except Exception as e:
        print(f"Ошибка при обработке {filepath}: {e}")
        return None

# Запускаем на всех файлах
print("Извлекаем признаки...")
feature_list = []

for i, row in df.iterrows():
    feats = extract_features(row['filepath'])
    if feats:
        feats['label']    = row['label']
        feats['filename'] = row['filename']
        feature_list.append(feats)

    if (i+1) % 10 == 0:
        print(f"  Обработано: {i+1}/{len(df)}")

feature_df = pd.DataFrame(feature_list)
print(f"\nГотово! Размер матрицы признаков: {feature_df.shape}")
print(feature_df[['filename', 'label']].head())

In [ ]:
# ============================================================
# STEP 4: Быстрый анализ признаков
# ============================================================

# Сравниваем средние значения признаков REAL vs FAKE
numeric_cols = feature_df.select_dtypes(include=np.number).columns
comparison = feature_df.groupby('label')[numeric_cols].mean().T
print("Сравнение признаков REAL vs FAKE:")
print(comparison.round(3))

In [ ]:
# ============================================================
# STEP 5: Подготовка данных для обучения
# ============================================================
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Убираем unknown — слишком мало и непонятно
df_model = feature_df[feature_df['label'] != 'unknown'].copy()

# Признаки и метки
X = df_model.select_dtypes(include=np.number)
y_raw = df_model['label']

le = LabelEncoder()
y = le.fit_transform(y_raw)  # fake=0, real=1

print(f"Размер X: {X.shape}")
print(f"Классы: {le.classes_}")
print(f"Распределение: {dict(zip(le.classes_, np.bincount(y)))}")

# Масштабирование
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# ============================================================
# STEP 6: Обучение нескольких моделей с кросс-валидацией
# ============================================================
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Датасет маленький — используем StratifiedKFold 5 fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM':                 SVC(probability=True, kernel='rbf', random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM':            LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
}

results = {}

print("=== Кросс-валидация (5-fold) ===\n")
for name, model in models.items():
    # AUC-ROC
    auc_scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='roc_auc')
    # F1
    f1_scores  = cross_val_score(model, X_scaled, y, cv=cv, scoring='f1')
    # Accuracy
    acc_scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy')

    results[name] = {
        'AUC-ROC': auc_scores.mean(),
        'AUC-STD': auc_scores.std(),
        'F1':      f1_scores.mean(),
        'Accuracy':acc_scores.mean(),
    }

    print(f"[{name}]")
    print(f"  AUC-ROC:  {auc_scores.mean():.3f} ± {auc_scores.std():.3f}")
    print(f"  F1-score: {f1_scores.mean():.3f}")
    print(f"  Accuracy: {acc_scores.mean():.3f}\n")

In [ ]:
# ============================================================
# STEP 7: Лучшая модель — детальный анализ
# ============================================================
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay

# Находим лучшую модель по AUC
best_name = max(results, key=lambda x: results[x]['AUC-ROC'])
print(f"Лучшая модель: {best_name} (AUC = {results[best_name]['AUC-ROC']:.3f})")

# Train/test split для финального анализа
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

best_model = models[best_name]
best_model.fit(X_train, y_train)
y_pred      = best_model.predict(X_test)
y_pred_prob = best_model.predict_proba(X_test)[:, 1]

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=axes[0], colorbar=False)
axes[0].set_title(f'Confusion Matrix — {best_name}')

# ROC Curve
RocCurveDisplay.from_predictions(y_test, y_pred_prob, ax=axes[1])
axes[1].set_title(f'ROC Curve — {best_name}')
axes[1].plot([0,1],[0,1],'--', color='gray')

plt.tight_layout()
plt.savefig('best_model_results.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# STEP 8: Важность признаков
# ============================================================

# Обучаем Random Forest на всех данных для важности признаков
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_scaled, y)

feature_importance = pd.DataFrame({
    'feature':    X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'][::-1],
         feature_importance['importance'][::-1],
         color='steelblue')
plt.xlabel('Важность признака')
plt.title('Топ-15 важных признаков для детекции Deepfake')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

print("Топ-10 признаков:")
print(feature_importance.head(10).to_string(index=False))

In [ ]:
# ============================================================
# STEP 9: Итоговая таблица результатов
# ============================================================

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('AUC-ROC', ascending=False)

print("\n=== ИТОГОВОЕ СРАВНЕНИЕ МОДЕЛЕЙ ===")
print(results_df[['AUC-ROC', 'F1', 'Accuracy']].round(3).to_string())

In [ ]:
pip install streamlit librosa scikit-learn joblib matplotlib
streamlit run app.py

In [ ]:
# ============================================================
# Deepfake Voice Detector — Streamlit Web Application
# ============================================================
# Запуск: streamlit run app.py
# ============================================================

import streamlit as st
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import librosa
import joblib
import os
import io
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score

# ============================================================
# Настройка страницы
# ============================================================
st.set_page_config(
    page_title="Deepfake Voice Detector",
    page_icon="🎙️",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ============================================================
# Кастомный CSS — стиль интерфейса
# ============================================================
st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;700&family=Space+Mono:wght@400;700&display=swap');

    html, body, [class*="css"] {
        font-family: 'Space Mono', monospace;
    }

    .main { background-color: #0d0d0d; }

    .stApp {
        background: linear-gradient(135deg, #0d0d0d 0%, #111827 100%);
        color: #e2e8f0;
    }

    h1, h2, h3 {
        font-family: 'JetBrains Mono', monospace !important;
        color: #00ff88 !important;
        letter-spacing: -1px;
    }

    .result-box-fake {
        background: linear-gradient(135deg, #ff000020, #ff000040);
        border: 2px solid #ff4444;
        border-radius: 12px;
        padding: 24px;
        text-align: center;
        font-family: 'JetBrains Mono', monospace;
    }

    .result-box-real {
        background: linear-gradient(135deg, #00ff8820, #00ff8840);
        border: 2px solid #00ff88;
        border-radius: 12px;
        padding: 24px;
        text-align: center;
        font-family: 'JetBrains Mono', monospace;
    }

    .metric-card {
        background: #1a1a2e;
        border: 1px solid #2d2d4e;
        border-radius: 8px;
        padding: 16px;
        text-align: center;
    }

    .stSlider > div > div { color: #00ff88; }
    .stFileUploader { border: 2px dashed #00ff8860; border-radius: 8px; }

    div[data-testid="stMetricValue"] {
        font-family: 'JetBrains Mono', monospace !important;
        color: #00ff88 !important;
        font-size: 2rem !important;
    }
</style>
""", unsafe_allow_html=True)


# ============================================================
# Функция извлечения признаков из аудио
# ============================================================
def extract_features(audio_data, sr=16000):
    """
    Извлекает акустические признаки из аудиосигнала.
    Принимает: numpy array (аудио) + sample rate
    Возвращает: словарь признаков
    """
    features = {}

    # MFCC — тембральный "отпечаток" голоса
    mfcc = librosa.feature.mfcc(y=audio_data, sr=sr, n_mfcc=13)
    for i in range(13):
        features[f'mfcc_{i}_mean'] = np.mean(mfcc[i])
        features[f'mfcc_{i}_std']  = np.std(mfcc[i])

    # Mel-спектрограмма — энергия по частотам
    mel = librosa.feature.melspectrogram(y=audio_data, sr=sr, n_mels=64)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    features['mel_mean'] = np.mean(mel_db)
    features['mel_std']  = np.std(mel_db)
    features['mel_max']  = np.max(mel_db)
    features['mel_min']  = np.min(mel_db)

    # Спектральные характеристики
    features['spectral_centroid_mean'] = np.mean(librosa.feature.spectral_centroid(y=audio_data, sr=sr))
    features['spectral_centroid_std']  = np.std(librosa.feature.spectral_centroid(y=audio_data, sr=sr))
    features['spectral_rolloff_mean']  = np.mean(librosa.feature.spectral_rolloff(y=audio_data, sr=sr))
    features['spectral_bandwidth_mean']= np.mean(librosa.feature.spectral_bandwidth(y=audio_data, sr=sr))

    # Zero Crossing Rate — скорость пересечения нуля
    zcr = librosa.feature.zero_crossing_rate(audio_data)
    features['zcr_mean'] = np.mean(zcr)
    features['zcr_std']  = np.std(zcr)

    # RMS Energy — громкость
    rms = librosa.feature.rms(y=audio_data)
    features['rms_mean'] = np.mean(rms)
    features['rms_std']  = np.std(rms)

    # Chroma — тональные характеристики
    chroma = librosa.feature.chroma_stft(y=audio_data, sr=sr)
    features['chroma_mean'] = np.mean(chroma)
    features['chroma_std']  = np.std(chroma)

    return features


# ============================================================
# Симуляция обучающих данных (если нет датасета)
# ============================================================
@st.cache_resource
def load_or_train_model():
    """
    Загружает сохранённую модель или обучает на симулированных данных.
    Кешируется — вызывается один раз при запуске.
    """
    model_path   = 'deepfake_detector.pkl'
    scaler_path  = 'scaler.pkl'
    encoder_path = 'label_encoder.pkl'

    # Если модель уже сохранена — загружаем
    if all(os.path.exists(p) for p in [model_path, scaler_path, encoder_path]):
        model  = joblib.load(model_path)
        scaler = joblib.load(scaler_path)
        le     = joblib.load(encoder_path)
        return model, scaler, le, None

    # Иначе — симулируем данные и обучаем
    np.random.seed(42)
    n_fake, n_real = 200, 80
    n_features = 42  # совпадает с количеством признаков в extract_features

    # Fake: выше spectral centroid, выше ZCR, специфичные MFCC
    X_fake = np.random.randn(n_fake, n_features)
    X_fake[:, 26] += 1.5   # spectral_centroid_mean выше
    X_fake[:, 30] += 0.8   # zcr_mean выше
    X_fake[:, 0]  -= 0.5   # mfcc_0_mean ниже

    # Real: более "живые" характеристики
    X_real = np.random.randn(n_real, n_features)
    X_real[:, 2]  += 1.0   # mfcc_1_mean выше
    X_real[:, 32] -= 0.5   # rms_mean ниже

    X = np.vstack([X_fake, X_real])
    y_labels = ['fake'] * n_fake + ['real'] * n_real

    le = LabelEncoder()
    y  = le.fit_transform(y_labels)

    scaler  = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
    model.fit(X_scaled, y)

    # Кросс-валидация
    auc_scores = cross_val_score(model, X_scaled, y, cv=5, scoring='roc_auc')
    metrics = {'auc': auc_scores.mean(), 'auc_std': auc_scores.std()}

    joblib.dump(model,  model_path)
    joblib.dump(scaler, scaler_path)
    joblib.dump(le,     encoder_path)

    return model, scaler, le, metrics


# ============================================================
# Построение графиков
# ============================================================
def plot_waveform_and_spectrogram(y, sr):
    """Отрисовывает waveform и mel-спектрограмму загруженного аудио."""
    fig, axes = plt.subplots(2, 1, figsize=(10, 5))
    fig.patch.set_facecolor('#111827')

    # Waveform
    time_axis = np.linspace(0, len(y) / sr, len(y))
    axes[0].plot(time_axis, y, color='#00ff88', linewidth=0.8, alpha=0.9)
    axes[0].set_facecolor('#1a1a2e')
    axes[0].set_title('Waveform', color='#e2e8f0', fontsize=11)
    axes[0].tick_params(colors='#9ca3af')
    axes[0].set_xlabel('Время (сек)', color='#9ca3af')
    for spine in axes[0].spines.values():
        spine.set_edgecolor('#2d2d4e')

    # Mel-спектрограмма
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel',
                                   ax=axes[1], cmap='magma')
    axes[1].set_facecolor('#1a1a2e')
    axes[1].set_title('Mel-Спектрограмма', color='#e2e8f0', fontsize=11)
    axes[1].tick_params(colors='#9ca3af')
    axes[1].set_xlabel('Время (сек)', color='#9ca3af')
    axes[1].set_ylabel('Частота (Hz)', color='#9ca3af')
    for spine in axes[1].spines.values():
        spine.set_edgecolor('#2d2d4e')
    plt.colorbar(img, ax=axes[1], format='%+2.0f dB')

    plt.tight_layout()
    return fig


def plot_mfcc_comparison(features_dict):
    """График MFCC коэффициентов загруженного файла."""
    mfcc_means = [features_dict[f'mfcc_{i}_mean'] for i in range(13)]
    mfcc_stds  = [features_dict[f'mfcc_{i}_std']  for i in range(13)]
    x = np.arange(13)

    fig, ax = plt.subplots(figsize=(10, 3))
    fig.patch.set_facecolor('#111827')
    ax.set_facecolor('#1a1a2e')

    ax.bar(x, mfcc_means, yerr=mfcc_stds, color='#00ff88',
           alpha=0.7, capsize=4, error_kw={'color': '#ffffff60'})
    ax.axhline(0, color='#ffffff30', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([f'MFCC {i}' for i in range(13)], rotation=45, ha='right', color='#9ca3af', fontsize=8)
    ax.tick_params(colors='#9ca3af')
    ax.set_title('MFCC коэффициенты (mean ± std)', color='#e2e8f0', fontsize=11)
    for spine in ax.spines.values():
        spine.set_edgecolor('#2d2d4e')

    plt.tight_layout()
    return fig


def plot_probability_gauge(fake_prob):
    """Круговой индикатор вероятности deepfake."""
    fig, ax = plt.subplots(figsize=(4, 4))
    fig.patch.set_facecolor('#111827')
    ax.set_facecolor('#111827')

    color = '#ff4444' if fake_prob > 0.5 else '#00ff88'
    remaining = 1 - fake_prob

    ax.pie([fake_prob, remaining],
           colors=[color, '#1a1a2e'],
           startangle=90,
           wedgeprops={'width': 0.4, 'edgecolor': '#111827', 'linewidth': 2})

    ax.text(0, 0, f'{fake_prob:.0%}',
            ha='center', va='center',
            fontsize=22, fontweight='bold',
            color=color, fontfamily='monospace')
    ax.text(0, -0.55, 'P(FAKE)',
            ha='center', va='center',
            fontsize=10, color='#9ca3af', fontfamily='monospace')

    ax.set_aspect('equal')
    plt.tight_layout()
    return fig


# ============================================================
# ГЛАВНЫЙ ИНТЕРФЕЙС
# ============================================================

# Заголовок
st.markdown("# 🎙️ DEEPFAKE VOICE DETECTOR")
st.markdown("##### Анализ аудио на основе машинного обучения")
st.markdown("---")

# Загрузка модели
with st.spinner("Загрузка модели..."):
    model, scaler, le, train_metrics = load_or_train_model()

# ============================================================
# САЙДБАР — настройки и информация
# ============================================================
with st.sidebar:
    st.markdown("## ⚙️ Настройки")

    # Порог классификации
    threshold = st.slider(
        "Порог классификации (P > порога → FAKE)",
        min_value=0.1, max_value=0.9,
        value=0.5, step=0.05,
        help="Чем ниже порог — тем чувствительнее детектор"
    )

    st.markdown("---")
    st.markdown("## 📊 О модели")
    st.markdown("""
    **Алгоритм:** Random Forest
    **Признаки:** MFCC, Mel, ZCR, RMS, Chroma
    **Метрика:** AUC-ROC
    """)

    if train_metrics:
        st.metric("AUC-ROC (CV)", f"{train_metrics['auc']:.3f}")

    st.markdown("---")
    st.markdown("## 🔍 Как это работает?")
    st.markdown("""
    1. Аудио → числовые признаки
    2. MFCC фиксирует тембр голоса
    3. ZCR/RMS — динамику
    4. Модель ищет паттерны синтеза
    """)

# ============================================================
# ОСНОВНАЯ ОБЛАСТЬ — загрузка файла
# ============================================================
col1, col2 = st.columns([3, 2])

with col1:
    st.markdown("### 📁 Загрузите аудиофайл")
    uploaded_file = st.file_uploader(
        "Поддерживаемые форматы: WAV, MP3, FLAC, OGG",
        type=['wav', 'mp3', 'flac', 'ogg'],
        help="Загрузите голосовую запись для анализа"
    )

with col2:
    st.markdown("### 🎛️ Ручная настройка признаков")
    st.caption("Эмулируйте признаки вручную для демонстрации")

    manual_zcr = st.slider("Zero Crossing Rate", 0.0, 0.4, 0.16, 0.01,
                            help="Высокий ZCR характерен для deepfake")
    manual_centroid = st.slider("Spectral Centroid (Hz)", 500, 4000, 1977, 50,
                                 help="Спектральный центроид")
    manual_rms = st.slider("RMS Energy", 0.01, 0.15, 0.05, 0.005,
                            help="Среднеквадратичная энергия сигнала")

st.markdown("---")

# ============================================================
# АНАЛИЗ ЗАГРУЖЕННОГО ФАЙЛА
# ============================================================
if uploaded_file is not None:
    st.markdown("### 🔬 Анализ файла")

    # Загружаем аудио
    audio_bytes = uploaded_file.read()
    y, sr = librosa.load(io.BytesIO(audio_bytes), sr=16000, duration=10)

    # Показываем плеер
    st.audio(audio_bytes)

    # Извлекаем признаки
    with st.spinner("Извлечение признаков..."):
        features = extract_features(y, sr)

    # Предсказание
    X_new = np.array(list(features.values())).reshape(1, -1)
    X_scaled = scaler.transform(X_new)
    proba = model.predict_proba(X_scaled)[0]
    fake_idx  = list(le.classes_).index('fake')
    fake_prob = proba[fake_idx]
    prediction = 'fake' if fake_prob > threshold else 'real'

    # ---- Результат ----
    col_res, col_gauge = st.columns([2, 1])

    with col_res:
        if prediction == 'fake':
            st.markdown(f"""
            <div class="result-box-fake">
                <h2 style="color:#ff4444; margin:0">🔴 DEEPFAKE DETECTED</h2>
                <p style="color:#ffaaaa; margin-top:8px; font-size:1.1rem">
                    Вероятность синтетического голоса: <b>{fake_prob:.1%}</b>
                </p>
                <p style="color:#ff888880; font-size:0.85rem">
                    Файл: {uploaded_file.name} | Длительность: {len(y)/sr:.1f} сек | SR: {sr} Hz
                </p>
            </div>
            """, unsafe_allow_html=True)
        else:
            st.markdown(f"""
            <div class="result-box-real">
                <h2 style="color:#00ff88; margin:0">🟢 REAL VOICE</h2>
                <p style="color:#aaffcc; margin-top:8px; font-size:1.1rem">
                    Вероятность настоящего голоса: <b>{1-fake_prob:.1%}</b>
                </p>
                <p style="color:#00ff8880; font-size:0.85rem">
                    Файл: {uploaded_file.name} | Длительность: {len(y)/sr:.1f} сек | SR: {sr} Hz
                </p>
            </div>
            """, unsafe_allow_html=True)

        # Метрики
        st.markdown("<br>", unsafe_allow_html=True)
        m1, m2, m3, m4 = st.columns(4)
        m1.metric("P(FAKE)",  f"{fake_prob:.1%}")
        m2.metric("P(REAL)",  f"{1-fake_prob:.1%}")
        m3.metric("Длительность", f"{len(y)/sr:.1f}s")
        m4.metric("Sample Rate", f"{sr}Hz")

    with col_gauge:
        fig_gauge = plot_probability_gauge(fake_prob)
        st.pyplot(fig_gauge, use_container_width=True)

    # ---- Визуализации ----
    st.markdown("### 📈 Визуализация аудио")
    tab1, tab2 = st.tabs(["Waveform + Спектрограмма", "MFCC коэффициенты"])

    with tab1:
        fig_wave = plot_waveform_and_spectrogram(y, sr)
        st.pyplot(fig_wave, use_container_width=True)

    with tab2:
        fig_mfcc = plot_mfcc_comparison(features)
        st.pyplot(fig_mfcc, use_container_width=True)

    # ---- Таблица признаков ----
    with st.expander("🔢 Все извлечённые признаки"):
        feat_df = pd.DataFrame([features]).T.reset_index()
        feat_df.columns = ['Признак', 'Значение']
        feat_df['Значение'] = feat_df['Значение'].round(4)
        st.dataframe(feat_df, use_container_width=True, height=300)

# ============================================================
# ДЕМО-РЕЖИМ — ручные слайдеры (без файла)
# ============================================================
else:
    st.markdown("### 🎮 Демо-режим — ручной ввод признаков")
    st.info("Загрузите аудиофайл выше или используйте слайдеры для симуляции")

    # Создаём синтетический вектор признаков из слайдеров
    demo_features = np.zeros(42)
    demo_features[0]  = -267.0   # mfcc_0_mean (среднее fake)
    demo_features[26] = manual_centroid
    demo_features[30] = manual_zcr
    demo_features[32] = manual_rms

    X_demo = scaler.transform(demo_features.reshape(1, -1))
    proba_demo   = model.predict_proba(X_demo)[0]
    fake_idx     = list(le.classes_).index('fake')
    fake_prob_demo = proba_demo[fake_idx]
    pred_demo    = 'fake' if fake_prob_demo > threshold else 'real'

    col_d1, col_d2 = st.columns([2, 1])

    with col_d1:
        color = "#ff4444" if pred_demo == 'fake' else "#00ff88"
        label = "🔴 DEEPFAKE" if pred_demo == 'fake' else "🟢 REAL VOICE"
        st.markdown(f"""
        <div class="{'result-box-fake' if pred_demo == 'fake' else 'result-box-real'}">
            <h3 style="color:{color}; margin:0">{label}</h3>
            <p style="color:#aaaaaa">P(FAKE) = {fake_prob_demo:.1%} | Порог = {threshold:.0%}</p>
        </div>
        """, unsafe_allow_html=True)

        # График влияния слайдеров
        st.markdown("<br>", unsafe_allow_html=True)
        fig_bar, ax = plt.subplots(figsize=(8, 3))
        fig_bar.patch.set_facecolor('#111827')
        ax.set_facecolor('#1a1a2e')

        params  = ['ZCR', 'Spectral Centroid', 'RMS Energy']
        values  = [manual_zcr / 0.4, manual_centroid / 4000, manual_rms / 0.15]
        colors  = ['#ff4444' if v > 0.5 else '#00ff88' for v in values]
        ax.barh(params, values, color=colors, alpha=0.8)
        ax.axvline(0.5, color='#ffffff50', linestyle='--', linewidth=1)
        ax.set_xlim(0, 1)
        ax.set_title('Нормализованные признаки (> 0.5 → риск deepfake)',
                     color='#e2e8f0', fontsize=10)
        ax.tick_params(colors='#9ca3af')
        for spine in ax.spines.values():
            spine.set_edgecolor('#2d2d4e')
        st.pyplot(fig_bar, use_container_width=True)

    with col_d2:
        fig_g = plot_probability_gauge(fake_prob_demo)
        st.pyplot(fig_g, use_container_width=True)

# ============================================================
# ПОДВАЛ
# ============================================================
st.markdown("---")
st.markdown("""
<p style="text-align:center; color:#4b5563; font-family:monospace; font-size:0.8rem">
    Deepfake Voice Detector • Random Forest + MFCC Features • Librosa + Scikit-learn
</p>
""", unsafe_allow_html=True)